In [1]:
import os, shutil, random
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/antispoof_data')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
for s in ['train', 'val']:
    for c in ['0_spoof', '1_real']:
        (WORK_DIR / s / c).mkdir(parents=True, exist_ok=True)

# 1. Tìm 2 dataset
esp_p1 = next(d for d in INPUT_ROOT.rglob('train/person-001') if d.is_dir())
esp_root = esp_p1.parent.parent
esp_tr = [p for p in (esp_root / 'train').rglob('*.*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
esp_va = [p for p in (esp_root / 'val').rglob('*.*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]

celeba_base = None
for root, dirs, _ in os.walk(INPUT_ROOT):
    if len(Path(root).relative_to(INPUT_ROOT).parts) > 4:
        dirs[:] = []
        continue
    for d in dirs:
        if 'celeba' in d.lower():
            celeba_base = Path(root) / d
            break
    if celeba_base:
        break

# 2. Lấy rải đều tối đa 12 ảnh/thư mục để bao phủ >1.200 người & đủ mọi loại thiết bị/ánh sáng
tr_live, tr_spoof, va_live, va_spoof = [], [], [], []
random.seed(42)

def scan_diverse(cur, is_val=False, lbl=None):
    if len(tr_live) >= 12000 and len(tr_spoof) >= 12000 and len(va_live) >= 1000 and len(va_spoof) >= 1000:
        return
    subs, imgs = [], []
    with os.scandir(cur) as it:
        for e in it:
            nl = e.name.lower()
            if lbl and nl.endswith(('.jpg', '.jpeg', '.png')):
                imgs.append(e.path)
            elif e.is_dir(follow_symlinks=False):
                subs.append(e.path)
    if imgs and lbl:
        random.shuffle(imgs)
        tgt = (va_live if is_val else tr_live) if lbl == 'live' else (va_spoof if is_val else tr_spoof)
        lim = 1000 if is_val else 12000
        if len(tgt) < lim:
            tgt.extend(imgs[:min(12, lim - len(tgt))])
    random.shuffle(subs)
    for sd in subs:
        if len(tr_live) >= 12000 and len(tr_spoof) >= 12000 and len(va_live) >= 1000 and len(va_spoof) >= 1000:
            break
        sn = os.path.basename(sd).lower()
        nv = is_val or ('val' in sn) or ('test' in sn)
        nl = 'live' if sn in {'live', 'real', '0'} else ('spoof' if sn in {'spoof', 'fake', '1'} else lbl)
        scan_diverse(sd, nv, nl)

scan_diverse(str(celeba_base))
if len(va_live) < 300:
    va_live, tr_live = tr_live[-1000:], tr_live[:-1000]
if len(va_spoof) < 300:
    va_spoof, tr_spoof = tr_spoof[-1000:], tr_spoof[:-1000]

# 3. Tạo liên kết dữ liệu (bỏ ảnh sọc nhân tạo để model học đặc trưng thật)
for i, p in enumerate(tr_live[:10500]):
    os.symlink(p, WORK_DIR / f"train/1_real/c_{i:05d}{Path(p).suffix}")
for r in range(10):
    for i, p in enumerate(esp_tr):
        os.symlink(p, WORK_DIR / f"train/1_real/e{r}_{i:05d}{p.suffix}")

for i, p in enumerate(tr_spoof[:12000]):
    os.symlink(p, WORK_DIR / f"train/0_spoof/s_{i:05d}{Path(p).suffix}")

for i, p in enumerate(va_live[:800]):
    os.symlink(p, WORK_DIR / f"val/1_real/c_{i:05d}{Path(p).suffix}")
for i, p in enumerate(esp_va):
    os.symlink(p, WORK_DIR / f"val/1_real/e_{i:05d}{p.suffix}")
for i, p in enumerate(va_spoof[:800]):
    os.symlink(p, WORK_DIR / f"val/0_spoof/s_{i:05d}{Path(p).suffix}")

print("XONG!", len(list((WORK_DIR/'train/1_real').glob('*'))), "Real |", len(list((WORK_DIR/'train/0_spoof').glob('*'))), "Spoof")

XONG! 12100 Real | 12000 Spoof


In [2]:
!pip install -q timm
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('/kaggle/working/antispoof_data/train', transform=train_tf)
val_ds = datasets.ImageFolder('/kaggle/working/antispoof_data/val', transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

device = torch.device('cuda')
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2, drop_rate=0.3).to(device)

# Phạt nặng gấp 2.5 lần khi lọt ảnh giả (0_spoof)
class_weights = torch.tensor([2.5, 1.0], device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)
scaler = torch.amp.GradScaler('cuda')

best_bal_acc = 0.0
for epoch in range(1, 13):
    model.train()
    tr_loss, tr_total = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(imgs)
            loss = criterion(out, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item() * imgs.size(0)
        tr_total += imgs.size(0)

    model.eval()
    c0 = t0 = c1 = t1 = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(1)
            c0 += ((preds == 0) & (labels == 0)).sum().item()
            t0 += (labels == 0).sum().item()
            c1 += ((preds == 1) & (labels == 1)).sum().item()
            t1 += (labels == 1).sum().item()

    scheduler.step()
    acc_spoof = c0 / max(1, t0) * 100
    acc_real = c1 / max(1, t1) * 100
    bal_acc = (acc_spoof + acc_real) / 2
    print(f"Epoch {epoch:02d}/12 | Loss: {tr_loss/tr_total:.4f} | Spoof Acc: {acc_spoof:.2f}% | Real Acc: {acc_real:.2f}%")
    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        torch.save(model.state_dict(), '/kaggle/working/antispoof_best.pth')

print(f"Hoàn tất! Balanced Val Acc cao nhất: {best_bal_acc:.2f}%")

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/12 | Loss: 0.1978 | Spoof Acc: 82.50% | Real Acc: 99.05%
Epoch 02/12 | Loss: 0.0477 | Spoof Acc: 81.62% | Real Acc: 99.76%
Epoch 03/12 | Loss: 0.0295 | Spoof Acc: 82.50% | Real Acc: 99.29%
Epoch 04/12 | Loss: 0.0239 | Spoof Acc: 82.50% | Real Acc: 99.76%
Epoch 05/12 | Loss: 0.0138 | Spoof Acc: 84.00% | Real Acc: 99.29%
Epoch 06/12 | Loss: 0.0109 | Spoof Acc: 74.88% | Real Acc: 100.00%
Epoch 07/12 | Loss: 0.0071 | Spoof Acc: 79.75% | Real Acc: 99.88%
Epoch 08/12 | Loss: 0.0037 | Spoof Acc: 84.62% | Real Acc: 99.52%
Epoch 09/12 | Loss: 0.0029 | Spoof Acc: 85.00% | Real Acc: 99.76%
Epoch 10/12 | Loss: 0.0012 | Spoof Acc: 83.75% | Real Acc: 99.64%
Epoch 11/12 | Loss: 0.0012 | Spoof Acc: 84.00% | Real Acc: 99.64%
Epoch 12/12 | Loss: 0.0006 | Spoof Acc: 84.50% | Real Acc: 99.52%
Hoàn tất! Balanced Val Acc cao nhất: 92.38%


In [3]:
!pip install -q onnx onnxscript
import numpy as np
from pathlib import Path
from PIL import Image

model.load_state_dict(torch.load('/kaggle/working/antispoof_best.pth'))
model.eval()

class AntiSpoofONNX(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, x):
        return torch.softmax(self.net(x), dim=1)

export_model = AntiSpoofONNX(model).to(device).eval()
dummy = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    export_model, dummy, '/kaggle/working/antispoof.onnx',
    input_names=['input'], output_names=['probs'],
    dynamic_axes={'input': {0: 'batch'}, 'probs': {0: 'batch'}},
    opset_version=12,
    dynamo=False
)

@torch.no_grad()
def predict_real_prob(img_path):
    img = Image.open(img_path).convert('RGB')
    t = val_tf(img).unsqueeze(0).to(device)
    return float(export_model(t).cpu().numpy()[0][1])

esp_real = [predict_real_prob(p) for p in Path('/kaggle/working/antispoof_data/val/1_real').glob('e_*.*')]
celeba_real = [predict_real_prob(p) for p in list(Path('/kaggle/working/antispoof_data/val/1_real').glob('c_*.*'))[:300]]
celeba_spoof = [predict_real_prob(p) for p in list(Path('/kaggle/working/antispoof_data/val/0_spoof').glob('s_*.*'))[:300]]

print(f"\n1. ESP32 người nhà (Real): {np.mean(esp_real)*100:.2f}% (Đạt >0.65: {sum(x > 0.65 for x in esp_real)}/{len(esp_real)})")
print(f"2. CelebA thật (Real):     {np.mean(celeba_real)*100:.2f}% (Đạt >0.65: {sum(x > 0.65 for x in celeba_real)}/{len(celeba_real)})")
print(f"3. CelebA giả mạo (Spoof): {np.mean(celeba_spoof)*100:.2f}% (Chặn <0.65: {sum(x < 0.65 for x in celeba_spoof)}/{len(celeba_spoof)})")

/tmp/ipykernel_49/3586307514.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(



1. ESP32 người nhà (Real): 100.00% (Đạt >0.65: 40/40)
2. CelebA thật (Real):     99.67% (Đạt >0.65: 299/300)
3. CelebA giả mạo (Spoof): 15.92% (Chặn <0.65: 256/300)
